## End-to-end demo of CREST using MNIST data 

The entire notebook should run in < 60 seconds, and use ~600MB of memory.

In [1]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ../..

#### Load MNIST

In [2]:
import tensorflow as tf

# Load data and define constants
xy_train, xy_test = tf.keras.datasets.mnist.load_data()   
n_classes  = 10
patch_size = 28

#### Create the datasets

In [3]:
from crest.src.data.loading import StructuredDataset

# Set input/output key labels
INP = 'image'
OUT = 'class'

# Define the train/test datasets, as well as the array labels
ds_train = StructuredDataset(*xy_train, labels=[INP, OUT])
ds_test  = StructuredDataset(*xy_test,  labels=[INP, OUT])

#### Create the batchers

In [4]:
from crest.src.data import Batcher

# Note: workers=0 is faster for this, but may cause issues in a notebook
# since the background threads will run in the main process 
batch_kwargs = {
    'batch_size' : 200,            # 200 samples in each batch
    'features'   : [[INP], [OUT]], # [{'image': ...}, {'class': ...}]
    'repeat'     : True,           # Yield indefinitely
    'workers'    : 0,              # Number of background processes
}
batch_train = Batcher(ds_train, **batch_kwargs)
batch_test  = Batcher(ds_test,  **batch_kwargs)

#### Create the CREST graph

In [5]:
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.src.model import HierarchalTensorGraph as HTG

def crest_layer(layer):
    """ Wrap a tensorflow layer in a HierarchalTensorGraph """
    graph = HTG(lambda d: {OUT: layer(d[INP])}, name=layer.name)
    graph.input_features  = INP
    graph.output_features = {OUT: INP}
    return graph

# Define the tensorflow layers, and convert to HTG objects
layers = [crest_layer(layer) for layer in [
    Lambda(lambda x: tf.cast(x, tf.float32)),         # Cast dtype
    Lambda(lambda x: tf.expand_dims(x, axis=-1)),     # Add feature dimension
    Conv2D(10, 3, padding='same', activation='relu'), # 28x28x1 -> 28x28x10
    Flatten(),                                        # 28x28x10 -> 7840
    Dropout(0.3),                                     # 30% of data dropped
    Dense(n_classes, activation='softmax'),           # Class probabilities
]]

# Create the CREST graph: ensure outputs changed to 'class' and add edges
graph = HTG(name='MNIST')
list(map(graph.add_edge, ['input']+layers, layers+['output']))
graph.output_features = {INP: OUT}

#### Create input / output dictionaries

In [6]:
from tensorflow.keras import Input

# Create the input/output dictionaries
inputs  = {INP: Input(shape=(patch_size, patch_size))}
outputs = graph(inputs)
print(f'\n Inputs: {inputs}')
print(f'Outputs: {outputs}')


 Inputs: {'image': <KerasTensor: shape=(None, 28, 28) dtype=float32 (created by layer 'input_1')>}
Outputs: {'class': <KerasTensor: shape=(None, 10) dtype=float32 (created by layer 'dense')>}


#### Fit the TensorFlow model

In [7]:
from tensorflow.keras import Model

# Create the TensorFlow model and print the summary
model = Model(inputs, outputs)
model.summary()

# Compile and fit the model - should take ~30-60 seconds
model.compile('Adam', loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
model.fit(batch_train, validation_data=batch_test, **{
    'steps_per_epoch'  : 300, # 60k samples / 200 samples per batch
    'validation_steps' : 50,  # 10k samples / 200 samples per batch
    'epochs'           : 5,
    'verbose'          : 1,
})

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 28, 28)]          0         
                                                                 
 lambda (Lambda)             (None, 28, 28)            0         
                                                                 
 lambda_1 (Lambda)           (None, 28, 28, 1)         0         
                                                                 
 conv2d (Conv2D)             (None, 28, 28, 10)        100       
                                                                 
 flatten (Flatten)           (None, 7840)              0         
                                                                 
 dropout (Dropout)           (None, 7840)              0         
                                                                 
 dense (Dense)               (None, 10)                78410 

#### Evaluate the model

We can now give dictionaries directly to the graph object:

In [8]:
eval_sets = {
    'First test batch' : next(batch_test),
    ' Entire test set' : ({INP: xy_test[0]}, {OUT: xy_test[1]}),
}

for label, (x, y) in eval_sets.items():
    # Generate the predictions and convert to numpy
    pred = graph(x)
    pred = {k: v.numpy() for k,v in pred.items()}

    # Check shapes
    for d in [x, y, pred]:
        print({k: v.shape for k,v in d.items()})

    # Check accuracy
    acc = (pred[OUT].argmax(axis=-1) == y[OUT]).mean()
    print(f'{label}: {100 * acc:.2f}% correct\n')

# Close the batchers
batch_train.close()
batch_test.close()

print(f'Notebook runtime: {time.time()-start_time:.1f} seconds')

{'image': (200, 28, 28)}
{'class': (200,)}
{'class': (200, 10)}
First test batch: 96.00% correct

{'image': (10000, 28, 28)}
{'class': (10000,)}
{'class': (10000, 10)}
 Entire test set: 97.17% correct

Notebook runtime: 35.9 seconds


Accuracy on the full test set should be > ~97%, and runtime < 60 seconds (assuming all cells were run at once). 